# nnU-Net ResEnc + CH + CAM + ACE on ACDC (ablation study)

Same ablation design as the COVIDx CT notebook, now with a **residual-encoder nnU-Net (ResEnc)** backbone on **ACDC** (cardiac cine-MRI; RV, myocardium, LV).

| Variant | CH | CAM | ACE |
|---|---|---|---|
| nnU-Net ResEnc | | | |
| nnU-Net ResEnc+CH | x | | |
| nnU-Net ResEnc+CH+CAM | x | x | |
| nnU-Net ResEnc+CH+CAM+ACE | x | x | x |

`L_total = L_seg + 0.2 L_cls + 0.3 L_CAM + 0.5 L_ACE`

**Differences from the binary (COVIDx) notebook, because ACDC is multi-class (bg, RV, Myo, LV):**
- `L_seg = 0.5 Dice + 0.5 CE`: softmax Dice over the 3 foreground classes plus cross-entropy (the multi-class counterpart of BCE).
- **CH** predicts 3 image-level labels (RV / Myo / LV present in the slice, derived from the mask) with BCE; **CAM** is computed per class.
- **Uncertainty** `U` uses multi-class priors and is averaged over classes; the **ACE metric** averages over all 4 classes.
- Segmentation metrics are computed **per 3D volume** (patient x ED/ES frame), the usual ACDC convention, and averaged over volumes.

**Protocol:** patient-wise 70 / 10 / 20 split (train / val / test). Train -> training and the prior bank. Val -> checkpoint selection only. Test -> evaluated once per variant after training. No patient appears in two splits.

**Runtime:** Runtime > Change runtime type > GPU.

## 1. Setup and configuration

In [ ]:
!pip -q install kagglehub nibabel h5py

import os, re, glob, random, math, time, warnings
import numpy as np, pandas as pd
from collections import Counter
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

device = "cuda" if torch.cuda.is_available() else "cpu"
AMP = device == "cuda"

# ---------------- configuration ----------------
SEED = 42
S = 256                       # slices are padded to square, then resized to S x S
BATCH = 12
EPOCHS = 30
LR = 1e-2                     # nnU-Net default: SGD, Nesterov, momentum 0.99, poly decay
SPLIT = (0.7, 0.1, 0.2)       # patient-wise train / val / test

C = 4                         # bg, RV, Myo, LV
CLASS_NAMES = ["bg", "RV", "Myo", "LV"]
NF = C - 1                    # foreground classes (image-level labels / CAM channels)

LAMBDA_CLS, LAMBDA_CAM, LAMBDA_ACE = 0.2, 0.3, 0.5
M_PRIORS = 5                  # M retrieved anatomical priors
PRIOR_RES = 64                # resolution for priors / uncertainty
ACE_REGION = 4                # regional calibration loss over 4x4 cells at PRIOR_RES
N_BINS = 10                   # bins for the ACE metric

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)
print("device:", device)

## 2. Data

The notebook first lists what is inside the Kaggle download, then auto-detects one of two common ACDC layouts:
1. **NIfTI**: `patientXXX_frameYY.nii.gz` with `patientXXX_frameYY_gt.nii.gz` (original ACDC layout);
2. **HDF5**: `.h5` files with an image key (`image`/`img`/`data`) and a label key (`label`/`mask`/`gt`/`seg`), either per-slice or per-volume.

I could not inspect this particular Kaggle upload, so if the printed file list shows another layout (png / npy), edit `collect_slices()` in the cell below; everything after it only needs a list of `(image_slice, mask_slice, volume_id, patient_id)`.

In [ ]:
import kagglehub
root = kagglehub.dataset_download("anhoangvo/acdc-dataset")
all_files = sorted(os.path.join(dp, f) for dp, _, fs in os.walk(root) for f in fs)
ext = lambda f: os.path.splitext(f[:-3] if f.endswith(".gz") else f)[1].lower()
print(root); print(Counter(ext(f) for f in all_files)); print(*[os.path.relpath(f, root) for f in all_files[:8]], sep="\n")

In [ ]:
import nibabel as nib

def pid_of(s):
    m = re.search(r"patient(\d+)", s); return m.group(1) if m else s

def to_square(a):
    h, w = a.shape; m = max(h, w); out = np.zeros((m, m), a.dtype)
    y0, x0 = (m - h) // 2, (m - w) // 2; out[y0:y0 + h, x0:x0 + w] = a; return out

def prep_img(sl, lo, hi):
    sl = (np.clip((sl - lo) / (hi - lo + 1e-8), 0, 1) * 255).astype(np.uint8)
    return np.asarray(Image.fromarray(to_square(sl)).resize((S, S), Image.BILINEAR))

def prep_msk(sl):
    return np.asarray(Image.fromarray(to_square(sl.astype(np.uint8))).resize((S, S), Image.NEAREST))

def slices_from_volume(v, g, vid, pid, axis):
    v, g = np.moveaxis(v, axis, 0), np.moveaxis(g, axis, 0)
    lo, hi = np.percentile(v, [0.5, 99.5])
    return [(prep_img(v[z], lo, hi), prep_msk(g[z]), vid, pid) for z in range(v.shape[0])]

def collect_slices():
    items = []
    gts = [f for f in all_files if f.endswith(("_gt.nii.gz", "_gt.nii"))]
    h5s = [f for f in all_files if f.endswith((".h5", ".hdf5"))]
    if gts:
        print("layout: NIfTI,", len(gts), "labelled volumes")
        for g in tqdm(gts):
            ip = g.replace("_gt.nii", ".nii")
            if not os.path.exists(ip): continue
            v = np.asarray(nib.load(ip).dataobj).astype(np.float32); gg = np.asarray(nib.load(g).dataobj).astype(np.int64)
            items += slices_from_volume(v, gg, os.path.basename(ip).split(".nii")[0], pid_of(g), axis=2)
    elif h5s:
        import h5py
        print("layout: HDF5,", len(h5s), "files")
        for f in tqdm(h5s):
            with h5py.File(f, "r") as h:
                ik = next((k for k in ("image", "img", "data") if k in h), None); lk = next((k for k in ("label", "mask", "gt", "seg") if k in h), None)
                if ik is None or lk is None: continue
                im, lb = np.asarray(h[ik][:]).astype(np.float32), np.asarray(h[lk][:]).astype(np.int64)
            stem = os.path.splitext(os.path.basename(f))[0]; vid = re.sub(r"_slice.*$", "", stem)
            if im.ndim == 2:
                items.append((prep_img(im, *np.percentile(im, [0.5, 99.5])), prep_msk(lb), vid, pid_of(stem)))
            else:                                                 # volume: assume the slice axis is the shortest one
                items += slices_from_volume(im, lb, vid, pid_of(stem), axis=int(np.argmin(im.shape)))
    else:
        raise RuntimeError("Unknown layout - see the file list above and adapt collect_slices().")
    return items

items = collect_slices()
labs = np.unique(np.concatenate([np.unique(i[1]) for i in items[::5]]))
print("slices:", len(items), "| label values:", labs)
if not set(labs.tolist()) <= {0, 1, 2, 3}:                     # remap e.g. {0,85,170,255}
    assert len(labs) == C, "unexpected label set"
    lut = {int(v): k for k, v in enumerate(labs)}
    items = [(a, np.vectorize(lut.get)(m).astype(np.uint8), v, p) for a, m, v, p in items]

In [ ]:
# ---- patient-wise split ----
pids = sorted({i[3] for i in items}); rng = np.random.RandomState(SEED); rng.shuffle(pids)
n = len(pids); n_tr, n_va = int(SPLIT[0] * n), int(SPLIT[1] * n)
split_of = {p: ("train" if k < n_tr else "val" if k < n_tr + n_va else "test") for k, p in enumerate(pids)}

def pack(its):
    imgs = torch.from_numpy(np.stack([i[0] for i in its])); msk = torch.from_numpy(np.stack([i[1] for i in its]))
    vs = [i[2] for i in its]; uv = {v: k for k, v in enumerate(dict.fromkeys(vs))}
    vid = torch.tensor([uv[v] for v in vs])
    cls = torch.stack([(msk == k).flatten(1).any(1) for k in range(1, C)], 1).float()   # image-level targets from the mask
    return imgs, msk, cls, vid, len(uv)

arrs, nvol = {}, {}
for sp in ("train", "val", "test"):
    a = pack([i for i in items if split_of[i[3]] == sp]); arrs[sp] = a[:4]; nvol[sp] = a[4]
    print(f"{sp}: {len(a[0])} slices, {a[4]} volumes, {sum(split_of[p] == sp for p in pids)} patients, image-level positives (RV/Myo/LV): {a[2].sum(0).int().tolist()}")

class SegDS(Dataset):
    def __init__(self, a, train=False): self.imgs, self.masks, self.cls, self.vid = a; self.train = train
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i):
        x = self.imgs[i].float().div(255.)[None]; y = self.masks[i].long()
        if self.train:
            if random.random() < 0.5: x, y = x.flip(-1), y.flip(-1)
            if random.random() < 0.5: x, y = x.flip(-2), y.flip(-2)
            x = x * (1 + 0.2 * (random.random() - 0.5)) + 0.1 * (random.random() - 0.5)
        x = (x - x.mean()) / (x.std() + 1e-6)
        return x, y, self.cls[i], i, self.vid[i]

train_ds, val_ds, test_ds = SegDS(arrs["train"], True), SegDS(arrs["val"]), SegDS(arrs["test"])
mk = lambda ds, sh: DataLoader(ds, BATCH, shuffle=sh, num_workers=2, drop_last=sh, pin_memory=True)
train_loader, val_loader, test_loader = mk(train_ds, True), mk(val_ds, False), mk(test_ds, False)

## 3. Model: nnU-Net ResEnc with optional CH and CAM

Residual-encoder nnU-Net in the style of the nnU-Net v2 ResEnc-M preset: a conv stem, then six stages of residual basic blocks (1-3-4-6-6-6 blocks; 32-64-128-256-320-320 features; InstanceNorm + LeakyReLU; strided first block per stage), and a plain decoder (transposed-conv upsampling, skip concatenation, one conv per stage). This is a reimplementation of the architecture, not the official nnU-Net package (no automatic planning, deep supervision or full augmentation suite).

- **CH:** GAP + linear layer on the bottleneck, giving 3 logits (RV, Myo, LV present).
- **CAM:** `CAM_k = ReLU(sum_c w_kc F_c)` per class, max-normalised and upsampled. For the CAM variants it is fed to the decoder as extra channels (scaled by the predicted image-level probability of each class) and supervised by `L_CAM`.

In [ ]:
def cba(i, o, s=1, act=True):
    l = [nn.Conv2d(i, o, 3, s, 1, bias=False), nn.InstanceNorm2d(o, affine=True)]
    return nn.Sequential(*l, nn.LeakyReLU(0.01, inplace=True)) if act else nn.Sequential(*l)

class ResBlock(nn.Module):                       # nnU-Net BasicBlockD
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.c1, self.c2 = cba(cin, cout, stride), cba(cout, cout, 1, act=False)
        self.skip = None if (cin == cout and stride == 1) else nn.Sequential(
            nn.Conv2d(cin, cout, 1, stride, bias=False), nn.InstanceNorm2d(cout, affine=True))
        self.act = nn.LeakyReLU(0.01, inplace=True)
    def forward(self, x):
        r = x if self.skip is None else self.skip(x)
        return self.act(self.c2(self.c1(x)) + r)

class ResEncUNet2D(nn.Module):
    def __init__(self, use_cls=False, use_cam=False, feats=(32, 64, 128, 256, 320, 320), blocks=(1, 3, 4, 6, 6, 6), n_cls=C):
        super().__init__()
        assert use_cls or not use_cam, "CAM needs the classification head"
        self.use_cls, self.use_cam = use_cls, use_cam
        n = len(feats)
        self.stem = cba(1, feats[0])
        stages = []
        for i in range(n):
            cin = feats[0] if i == 0 else feats[i - 1]
            stages.append(nn.Sequential(*[ResBlock(cin if b == 0 else feats[i], feats[i], (1 if i == 0 else 2) if b == 0 else 1) for b in range(blocks[i])]))
        self.enc = nn.ModuleList(stages)
        self.ups = nn.ModuleList([nn.ConvTranspose2d(feats[i + 1], feats[i], 2, 2) for i in range(n - 2, -1, -1)])
        self.decs = nn.ModuleList([cba(2 * feats[i], feats[i]) for i in range(n - 2, -1, -1)])
        if use_cls: self.fc = nn.Linear(feats[-1], NF)
        if use_cam: self.refine = cba(feats[0] + NF, feats[0])
        self.head = nn.Conv2d(feats[0], n_cls, 1)

    def forward(self, x):
        size = x.shape[-2:]; x = self.stem(x); skips = []
        for e in self.enc:
            x = e(x); skips.append(x)
        bott = skips[-1]; d = bott
        for j, (up, dec) in enumerate(zip(self.ups, self.decs)):
            d = dec(torch.cat([up(d), skips[len(skips) - 2 - j]], 1))
        out = {}
        if self.use_cls:
            logit = self.fc(bott.mean((2, 3)))                                          # (B,3)
            cam = F.relu(torch.einsum("bchw,kc->bkhw", bott, self.fc.weight))           # (B,3,h,w)
            cam = cam / (cam.amax((2, 3), keepdim=True) + 1e-6)
            cam = F.interpolate(cam, size=size, mode="bilinear", align_corners=False)
            out["cls"], out["cam"] = logit, cam
            if self.use_cam:
                d = self.refine(torch.cat([d, cam * torch.sigmoid(logit)[:, :, None, None]], 1))
        out["logits"] = self.head(d)
        return out

_m = ResEncUNet2D(True, True).to(device)
with torch.no_grad(): _o = _m(torch.randn(2, 1, S, S, device=device))
print({k: tuple(v.shape) for k, v in _o.items()}, "| params: %.1fM" % (sum(p.numel() for p in _m.parameters()) / 1e6))
del _m, _o

## 4. Losses, anatomy-aware uncertainty, calibration

**Prior bank:** training-set ground-truth label maps (one-hot, down-sampled to 64x64). For each prediction `y*` (softmax probabilities), the `M` most similar priors are retrieved by cosine similarity over the foreground-class maps, and

`U = (1/M) * sum_i w_i * |y_bar_i - y*|`, averaged over the 4 classes, `w_i` = clipped cosine similarity.

An image's own mask is excluded during training. Retrieval is non-differentiable; gradients flow through `y*`.

**L_ACE** (regional): `U` and the segmentation error `e = mean_c |y*_c - y_c|` (detached target) are averaged over `ACE_REGION x ACE_REGION` cells, then `L_ACE = mean |u_i - e_i|`.

**ACE metric:** mean `|observed accuracy - expected confidence|` over the 4 classes and non-empty confidence bins.

In [ ]:
def onehot(y):                                         # (B,H,W) long -> (B,C,H,W) float
    return F.one_hot(y, C).permute(0, 3, 1, 2).float()

def seg_loss(logits, y):                               # L_seg = 1/2 Dice + 1/2 CE
    p = torch.softmax(logits, 1)[:, 1:]; oh = onehot(y)[:, 1:]
    inter = (p * oh).sum((0, 2, 3)); den = p.sum((0, 2, 3)) + oh.sum((0, 2, 3))
    dice = 1 - ((2 * inter + 1) / (den + 1)).mean()
    return 0.5 * dice + 0.5 * F.cross_entropy(logits, y)

def cam_loss(cam, y, c):                               # per-class CAM vs mask, only where that class is present in the slice
    oh = onehot(y)[:, 1:]
    bce = F.binary_cross_entropy(cam.clamp(1e-4, 1 - 1e-4), oh, reduction="none").mean((2, 3))   # (B,3)
    return (bce * c).sum() / c.sum().clamp(min=1)

class PriorBank:
    def __init__(self, masks, res=PRIOR_RES):
        self.res = res; chunks = []
        for i in range(0, len(masks), 128):
            oh = onehot(masks[i:i + 128].long())
            chunks.append(F.adaptive_avg_pool2d(oh, res).flatten(2))                  # (b,C,res*res)
        self.bank = torch.cat(chunks).to(device)                                      # (N,C,res*res)
        self.norm = F.normalize(self.bank[:, 1:].flatten(1), dim=1)                   # similarity on foreground maps
    @torch.no_grad()
    def retrieve(self, prob, M, exclude=None):
        q = F.adaptive_avg_pool2d(prob, self.res).flatten(2)[:, 1:].flatten(1)
        sim = F.normalize(q, dim=1) @ self.norm.T
        if exclude is not None: sim[torch.arange(len(q), device=sim.device), exclude.to(sim.device)] = -1
        w, idx = sim.topk(M, dim=1)
        return w.clamp(0, 1), self.bank[idx]                                          # (B,M), (B,M,C,res*res)

def uncertainty(prob, bank, M=M_PRIORS, exclude=None):
    prob = prob.float()
    w, pri = bank.retrieve(prob.detach(), M, exclude)
    q = F.adaptive_avg_pool2d(prob, bank.res).flatten(2)                              # (B,C,res*res)
    U = (w[:, :, None, None] * (pri - q[:, None]).abs()).sum(1) / M                   # (B,C,res*res)
    return U.mean(1).view(-1, 1, bank.res, bank.res)

def ace_loss(U, prob, y, res=PRIOR_RES, region=ACE_REGION):
    e = (F.adaptive_avg_pool2d(prob, res) - F.adaptive_avg_pool2d(onehot(y), res)).abs().mean(1, keepdim=True).detach()
    return (F.avg_pool2d(U, region) - F.avg_pool2d(e, region)).abs().mean()

bank = PriorBank(arrs["train"][1])                    # built from TRAINING masks only
print("prior bank:", tuple(bank.bank.shape))

## 5. Evaluation (per-volume Dice, IoU, Precision, Recall, F1, ACE, uncertainty magnitude and error correlation)

In [ ]:
def autocast(): return torch.autocast("cuda", enabled=AMP)

def confusion(pred, y, vid, tp, fp, fn):
    op, oy = F.one_hot(pred, C).permute(0, 3, 1, 2).bool(), F.one_hot(y, C).permute(0, 3, 1, 2).bool()
    tp.index_add_(0, vid, (op & oy).sum((2, 3)).double())
    fp.index_add_(0, vid, (op & ~oy).sum((2, 3)).double())
    fn.index_add_(0, vid, (~op & oy).sum((2, 3)).double())

def volume_metrics(tp, fp, fn):
    valid = (tp + fn) > 0
    d = 2 * tp / (2 * tp + fp + fn + 1e-9); iou = tp / (tp + fp + fn + 1e-9)
    pr = tp / (tp + fp + 1e-9); rc = tp / (tp + fn + 1e-9)
    per_class = lambda m: [m[valid[:, k], k].mean().item() for k in range(1, C)]
    return per_class(d), per_class(iou), per_class(pr), per_class(rc)

@torch.no_grad()
def val_dice(model, loader, nv):
    model.eval()
    tp, fp, fn = (torch.zeros(nv, C, dtype=torch.float64, device=device) for _ in range(3))
    for x, y, c, idx, vid in loader:
        x, y = x.to(device), y.to(device)
        with autocast(): out = model(x)
        confusion(out["logits"].argmax(1), y, vid.to(device), tp, fp, fn)
    return float(np.mean(volume_metrics(tp, fp, fn)[0]))

@torch.no_grad()
def evaluate(model, loader, bank, nv, n_vis=0):
    model.eval(); nb = N_BINS
    tp, fp, fn = (torch.zeros(nv, C, dtype=torch.float64, device=device) for _ in range(3))
    cnt = torch.zeros(C, nb, device=device, dtype=torch.float64); sconf = torch.zeros_like(cnt); sacc = torch.zeros_like(cnt)
    g = torch.zeros(6, dtype=torch.float64, device=device)      # n, sum u, sum e, sum u^2, sum e^2, sum ue
    uerr = uok = nerr = nok = 0.0
    pcorr, img_u, img_e, vis = [], [], [], []
    for x, y, c, idx, vid in loader:
        x, y = x.to(device), y.to(device)
        with autocast(): out = model(x)
        prob = torch.softmax(out["logits"].float(), 1); pred = prob.argmax(1)
        confusion(pred, y, vid.to(device), tp, fp, fn)
        for k in range(C):                                       # calibration bins per class
            conf, ind = prob[:, k], (y == k).double()
            b = (conf * nb).long().clamp(0, nb - 1).flatten()
            cnt[k] += torch.bincount(b, minlength=nb).double()
            sconf[k] += torch.bincount(b, weights=conf.flatten().double(), minlength=nb)
            sacc[k] += torch.bincount(b, weights=ind.flatten(), minlength=nb)
        U = uncertainty(prob, bank)                              # uncertainty vs. error
        err = F.adaptive_avg_pool2d((pred != y).float()[:, None], bank.res)
        u, e = U.flatten(1).double(), err.flatten(1).double()
        uc, ec = u - u.mean(1, keepdim=True), e - e.mean(1, keepdim=True)
        den = uc.norm(dim=1) * ec.norm(dim=1); ok = den > 1e-8
        pcorr += ((uc * ec).sum(1)[ok] / den[ok]).tolist()
        has = e > 0
        uerr += u[has].sum().item(); nerr += has.sum().item(); uok += u[~has].sum().item(); nok += (~has).sum().item()
        g += torch.stack([torch.tensor(float(u.numel()), device=device, dtype=torch.float64), u.sum(), e.sum(), (u * u).sum(), (e * e).sum(), (u * e).sum()])
        img_u += u.mean(1).tolist(); img_e += e.mean(1).tolist()
        if len(vis) < n_vis:
            cam = out.get("cam", None)
            for j in range(len(x)):
                if len(vis) >= n_vis: break
                if (y[j] > 0).sum() == 0: continue               # show slices that contain heart
                vis.append(dict(x=x[j, 0].cpu(), y=y[j].cpu(), p=pred[j].cpu(),
                                cam=None if cam is None else cam[j].float().cpu(),
                                U=F.interpolate(U[j:j + 1], size=x.shape[-2:], mode="bilinear")[0, 0].cpu()))
    n, su, se, suu, see, sue = g.tolist()
    cov = sue / n - (su / n) * (se / n)
    gcorr = cov / (math.sqrt(max(suu / n - (su / n) ** 2, 1e-12)) * math.sqrt(max(see / n - (se / n) ** 2, 1e-12)))
    valid = cnt > 0
    o, ex = sacc / cnt.clamp(min=1), sconf / cnt.clamp(min=1)
    ace = (((o - ex).abs() * valid).sum() / valid.sum()).item()
    d, iou, pr, rc = volume_metrics(tp, fp, fn)
    TP, FP, FN = tp[:, 1:].sum().item(), fp[:, 1:].sum().item(), fn[:, 1:].sum().item()
    res = {"Dice": np.mean(d), "Dice_RV": d[0], "Dice_Myo": d[1], "Dice_LV": d[2],
           "IoU": np.mean(iou), "Precision": np.mean(pr), "Recall": np.mean(rc),
           "F1": 2 * TP / (2 * TP + FP + FN + 1e-9),             # pooled over all foreground pixels
           "ACE": ace,
           "U_mean": su / n, "U_on_error": uerr / max(nerr, 1), "U_on_correct": uok / max(nok, 1),
           "corr_U_err_pixel": gcorr, "corr_U_err_per_image": float(np.mean(pcorr)) if pcorr else float("nan"),
           "corr_meanU_vs_errorrate": float(np.corrcoef(img_u, img_e)[0, 1])}
    return res, vis

## 6. Training

In [ ]:
CKPT_DIR = "/content/ckpt_acdc"; os.makedirs(CKPT_DIR, exist_ok=True)

VARIANTS = {   # name: (use_CH, use_CAM, use_ACE)
    "nnU-Net ResEnc":             (False, False, False),
    "nnU-Net ResEnc+CH":          (True,  False, False),
    "nnU-Net ResEnc+CH+CAM":      (True,  True,  False),
    "nnU-Net ResEnc+CH+CAM+ACE":  (True,  True,  True),
}

def train_variant(name, use_ch, use_cam, use_ace):
    set_seed(SEED)
    model = ResEncUNet2D(use_ch, use_cam).to(device)
    opt = torch.optim.SGD(model.parameters(), lr=LR, momentum=0.99, nesterov=True, weight_decay=3e-5)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    best, path, hist = -1, f"{CKPT_DIR}/{name.replace(' ', '_')}.pt", []
    for ep in range(EPOCHS):
        for gp in opt.param_groups: gp["lr"] = LR * (1 - ep / EPOCHS) ** 0.9          # poly schedule
        model.train(); agg = {"tot": 0, "seg": 0, "cls": 0, "cam": 0, "ace": 0}; nit = 0
        for x, y, c, idx, vid in tqdm(train_loader, desc=f"{name} ep{ep + 1}/{EPOCHS}", leave=False):
            x, y, c = x.to(device), y.to(device), c.to(device)
            with autocast(): out = model(x)
            logits = out["logits"].float()
            Ls = seg_loss(logits, y); L = Ls; Lc = Lm = La = torch.zeros((), device=device)
            if use_ch:
                Lc = F.binary_cross_entropy_with_logits(out["cls"].float(), c); L = L + LAMBDA_CLS * Lc
            if use_cam:
                Lm = cam_loss(out["cam"].float(), y, c); L = L + LAMBDA_CAM * Lm
            if use_ace:
                prob = torch.softmax(logits, 1)
                U = uncertainty(prob, bank, M_PRIORS, exclude=idx)
                La = ace_loss(U, prob, y); L = L + LAMBDA_ACE * La
            opt.zero_grad(set_to_none=True)
            scaler.scale(L).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 12)
            scaler.step(opt); scaler.update()
            for k, v in zip(agg, (L, Ls, Lc, Lm, La)): agg[k] += float(v)
            nit += 1
        vd = val_dice(model, val_loader, nvol["val"])
        hist.append({"epoch": ep + 1, "val_dice": vd, **{k: v / nit for k, v in agg.items()}})
        print(f"[{name}] ep{ep + 1:02d} loss {agg['tot'] / nit:.4f} (seg {agg['seg'] / nit:.3f} cls {agg['cls'] / nit:.3f} "
              f"cam {agg['cam'] / nit:.3f} ace {agg['ace'] / nit:.3f}) | val Dice {vd:.4f}")
        if vd > best: best = vd; torch.save(model.state_dict(), path)
    model.load_state_dict(torch.load(path))               # best-on-VALIDATION checkpoint
    return model, pd.DataFrame(hist)

models, histories, results, vis_store = {}, {}, {}, {}
for name, flags in VARIANTS.items():
    t0 = time.time()
    models[name], histories[name] = train_variant(name, *flags)
    results[name], vis_store[name] = evaluate(models[name], test_loader, bank, nvol["test"], n_vis=8)   # TEST: evaluated once, after training
    print(f"== {name}: done in {(time.time() - t0) / 60:.1f} min | test Dice {results[name]['Dice']:.4f}\n")

## 7. Results

In [ ]:
res_df = pd.DataFrame(results).T
seg_cols = ["Dice", "Dice_RV", "Dice_Myo", "Dice_LV", "IoU", "Precision", "Recall", "F1"]
unc_cols = ["ACE", "U_mean", "U_on_error", "U_on_correct", "corr_U_err_pixel", "corr_U_err_per_image", "corr_meanU_vs_errorrate"]
print("Segmentation (test, per-volume, mean over volumes)"); display(res_df[seg_cols].round(4))
print("Calibration / uncertainty (test)"); display(res_df[unc_cols].round(4))
res_df.to_csv("/content/acdc_ablation_results.csv")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for n_, h in histories.items():
    ax[0].plot(h.epoch, h.seg, label=n_); ax[1].plot(h.epoch, h.val_dice, label=n_)
ax[0].set_title("train L_seg"); ax[1].set_title("val Dice (volume-wise)"); ax[1].legend(fontsize=7)
for a in ax: a.set_xlabel("epoch")
plt.tight_layout(); plt.show()

## 8. Qualitative analysis: predictions, class activation maps, uncertainty maps

In [ ]:
seg_cmap = ListedColormap(["#000000", "#e41a1c", "#4daf4a", "#377eb8"])    # RV red, Myo green, LV blue

def overlay(a, ax_, m):
    ax_.imshow(a["x"], cmap="gray"); ax_.imshow(np.ma.masked_where(m == 0, m), cmap=seg_cmap, vmin=0, vmax=3, alpha=0.55, interpolation="nearest")

def show_qualitative(name, n=6):
    v = vis_store[name][:n]
    cols = ["Image", "Ground truth", "Prediction", "CAM (R=RV G=Myo B=LV)", "Uncertainty U", "Error"]
    fig, axs = plt.subplots(len(v), 6, figsize=(14, 2.3 * len(v))); axs = np.atleast_2d(axs)
    for r, s in enumerate(v):
        axs[r, 0].imshow(s["x"], cmap="gray")
        overlay(s, axs[r, 1], s["y"].numpy()); overlay(s, axs[r, 2], s["p"].numpy())
        axs[r, 3].imshow(s["x"], cmap="gray")
        if s["cam"] is not None: axs[r, 3].imshow(s["cam"].permute(1, 2, 0).clamp(0, 1).numpy(), alpha=0.6)
        im = axs[r, 4].imshow(s["U"], cmap="magma"); plt.colorbar(im, ax=axs[r, 4], fraction=0.046)
        axs[r, 5].imshow((s["p"] != s["y"]).float(), cmap="gray", vmin=0, vmax=1)
        for a in axs[r]: a.axis("off")
    for j, t in enumerate(cols): axs[0, j].set_title(t, fontsize=9)
    fig.suptitle(name); plt.tight_layout(); plt.show()

show_qualitative("nnU-Net ResEnc+CH+CAM+ACE")
# show_qualitative("nnU-Net ResEnc")   # baseline for comparison (no CAM panel)